In [1]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install openai
!pip install tiktoken

   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/874.7 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 874.7/874.7 kB 2.6 MB/s  0:00:00


In [1]:

# Imports

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS



In [3]:
# Loading the document

# Load PDF

pdf_path = r"C:\Users\KOWSIK.S\Downloads\MACHINE LEARNING(R17A0534).pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 120


In [5]:
# Check Content

print(documents[0].page_content[:1000])

MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100, Telangana State, India


In [7]:
# Chunking

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 315


In [9]:
# Embeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\KOWSIK.S\AppData\Local\Temp\ipykernel_12736\2987403494.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
# Create vectors database

vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)


print("Vector DB Created Successfully")

Vector DB Created Successfully


In [13]:
# Test Retrieval

query = "What is Machine Learning?"

results = vector_db.similarity_search(
    query,
    k=5
)

for doc in results:
    print(doc.page_content)
    print("="*100)

1 
 
UNIT I  
Introduction to Machine Learning 
1. Introduction 
 
1.1 What Is Machine Learning?  
Machine learning is programming computers to optimize a performance criterion using example 
data or past experience. We have a model defined up to some parameters, and learning is the 
execution of a computer program to optimize the parameters of the model using the training data or 
past experience. The model may be predictive to make predictions in the future, or descriptive to gain 
knowledge from data, or both. 
Arthur Samuel, an early American leader in the field of computer gaming and artificial intellige nce, 
coined the term “Machine Learning” in 1959 while at IBM. He defined machine learning as “the field of 
study that gives computers the ability to learn without being explicitly programmed.” However, there is 
no universally accepted definition for machine learning. Different authors define the term differently. 
 
Definition of learning 
Definition
Application of machine le a

In [15]:
# Connect LLM

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3",
    num_predict=300
)

In [17]:
# Create Context

context = "\n\n".join(
    [doc.page_content for doc in results]
)

In [19]:
prompt = f"""
You are a friendly and knowledgeable teacher.

Answer the question using ONLY the information provided in the context.

Instructions:
1. Start with a short introduction.
2. Explain every concept point by point.
3. For each point use the format:

Point 1:
- Explanation:
- Why it is important:
- Example:

Point 2:
- Explanation:
- Why it is important:
- Example:

4. Use simple language suitable for beginners.
5. Do not copy text directly from the context.
6. Explain in your own words.
7. If information is not available in the context, say:
   "I could not find that information in the uploaded document."

Context:
{context}

Question:
{query}

Answer:
"""

In [21]:
# Generate Answer

response = llm.invoke(prompt)

print(response)

Welcome to our discussion on Machine Learning!

Let's start with the definition of Machine Learning. According to Arthur Samuel, an early American leader in computer gaming and artificial intelligence, Machine Learning is "the field of study that gives computers the ability to learn without being explicitly programmed." This definition suggests that Machine Learning is about programming computers to optimize a performance criterion using example data or past experience.

**Point 1:**
- Explanation: Machine Learning is a way to train computers to make decisions or predictions without being told exactly how to do it.
- Why it's important: This allows computers to adapt and improve over time, making them more efficient and effective in various applications.
- Example: Imagine a computer program that can recognize faces and learn to identify new people based on the examples you provide.

**Point 2:**
- Explanation: Machine Learning can be used for both predictive (making predictions) and d

In [23]:
# Interactive Chat

while True:

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break

    results = vector_db.similarity_search(
        query,
        k=5
    )

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
You are a friendly and knowledgeable teacher.

Answer the question using ONLY the information provided in the context.

Instructions:
1. Start with a short introduction.
2. Explain every concept point by point.
3. For each point use the format:

Point 1:
- Explanation:
- Why it is important:
- Example:

Point 2:
- Explanation:
- Why it is important:
- Example:

4. Use simple language suitable for beginners.
5. Do not copy text directly from the context.
6. Explain in your own words.
7. If information is not available in the context, say:
   "I could not find that information in the uploaded document."

Context:
{context}

Question:
{query}

Answer:
"""
    response = llm.invoke(prompt)

    print("\nAnswer:")
    print(response)


Ask Question:  What is Machine Learning and it's types?



Answer:
Welcome! Today, we're going to explore the world of Machine Learning. Let's start with a brief introduction.

Machine learning is programming computers to optimize a performance criterion using example data or past experience. It's the process of training a computer program to make predictions or decisions based on data, without being explicitly programmed.

Now, let's dive deeper into the types of machine learning. According to our text, there are three main types:

**Point 1: Supervised Learning**
- Explanation: In supervised learning, we provide a training set of examples with correct responses (targets). The algorithm generalizes from this training set to respond correctly to all possible inputs.
- Why it is important: This type of learning allows us to train models that can make predictions or classify new data based on patterns learned from the training data.
- Example: Image classification, where a model learns to recognize different objects based on labeled images.

**

KeyboardInterrupt: Interrupted by user